# 4. Spectral grids, split grids and interpolation

This notebook loads installed grids and interpolates spectra from them. It covers:
1. a grid's spectra and parameters (`SpectralGrid`, `Spectrum`);
2. interpolation in Teff, log g and [M/H], and what happens outside the grid;
3. **split grids**: grids whose spectra differ in more than Teff, log g and [M/H] (Kurucz2003all's `alpha`, Husfeld's helium abundance `yhe`). Such a grid is stored as one folder per variant, plus an all-axes cube `flux_cube_nd.bin` that holds every variant with each extra axis as a further dimension;
4. writing a grid's spectra to another folder.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sed_tools.api import SED

plt.rcParams["figure.dpi"] = 80
SIGMA_SB = 5.670374419e-5
KURUCZ_CUT = dict(teff_min=4000, teff_max=8000, logg_min=3.0, logg_max=5.0,
                  metallicity_min=-1.0, metallicity_max=0.5)
SED.fetch("Kurucz2003all", **KURUCZ_CUT)
SED.fetch("Husfeld")
stellar_models = SED._model_root
print(stellar_models)

## 1. A grid's spectra and parameters

`SED.local(name)` loads an installed grid. `sed.grid` is its `SpectralGrid`: the spectra its `lookup_table.csv` lists, each a `Spectrum` with wavelength (Å), flux (erg s⁻¹ cm⁻² Å⁻¹), Teff, log g and [M/H].

In [ ]:
variant = SED.local("Kurucz2003all/Kurucz2003all_alpha_0p00_lh_1p25_vtur_2_km_s")
grid = variant.grid
print(len(grid), "spectra;", variant.parameter_ranges())
print("Teff:", grid.teff_grid)
print("log g:", grid.logg_grid, " [M/H]:", grid.metallicity_grid)
grid.parameters.head()

In [ ]:
cool = grid.filter(teff_max=5000, logg_min=4.5)
spectrum = cool[0]
print(len(cool), "spectra with Teff ≤ 5000 K and log g ≥ 4.5; first:", spectrum.filename,
      spectrum.teff, spectrum.logg, spectrum.metallicity, spectrum.wavelength.size, "points")

## 2. Interpolation

Calling the loaded grid interpolates its flux cube with cubic Hermite polynomials in Teff, log g and [M/H]. At a grid node it returns that node's spectrum.

In [ ]:
node = variant(teff=5750, logg=4.5, metallicity=0.0)
listed = next(s for s in grid if (s.teff, s.logg, s.metallicity) == (5750, 4.5, 0.0))
on_node = np.interp(listed.wavelength, node.wavelength, node.flux)
print("largest relative difference at a node:", np.max(np.abs(on_node - listed.flux) / listed.flux.max()))

fig, ax = plt.subplots(figsize=(7, 3.5))
for teff in (4000, 4900, 5777, 6600, 8000):
    s = variant(teff=teff, logg=4.4, metallicity=0.0)
    ratio = np.trapezoid(s.flux, s.wavelength) / (SIGMA_SB * teff**4)
    ax.loglog(s.wavelength, s.flux, lw=0.7, label=f"{teff} K (∫F/σT⁴ = {ratio:.4f})")
    assert abs(ratio - 1) < 5e-3
ax.set(xlim=(1500, 5e4), ylim=(1e3, 1e8), xlabel="Wavelength (Å)", ylabel="F$_\\lambda$")
ax.legend(fontsize=8);

Outside the grid, a value is clamped to the nearest edge (`fill_gaps=True`, the default). With `fill_gaps=False` it raises `ValueError`.

In [ ]:
clamped = variant(teff=9000, logg=4.5, metallicity=0.0)
print("asked 9000 K, got", clamped.teff)

strict = SED.local("Kurucz2003all/Kurucz2003all_alpha_0p00_lh_1p25_vtur_2_km_s", fill_gaps=False)
try:
    strict(teff=9000, logg=4.5, metallicity=0.0)
except ValueError as error:
    print("fill_gaps=False:", error)

## 3. Split grids and the all-axes cube

Kurucz2003all's spectra differ in `alpha` ([α/M] = 0 or 0.4) at each (Teff, log g, [M/H]) node. A flux cube that MESA reads has only those three axes, so SED_Tools splits the grid:
- one folder per variant, each with its own `flux_cube.bin` and `lookup_table.csv` (what MESA reads);
- `variants_index.csv` in the grid's folder lists them;
- `flux_cube_nd.bin` in the grid's folder holds every variant, with `alpha` as a fourth axis. SED_Tools reads it; MESA does not. The grid's folder has no `flux_cube.bin` of its own.

In [ ]:
kurucz_dir = stellar_models / "Kurucz2003all"
print(pd.read_csv(kurucz_dir / "variants_index.csv").to_string(index=False))
print(sorted(p.name for p in kurucz_dir.iterdir() if not p.name.endswith(".txt")))

`SED.local("Kurucz2003all")` loads the all-axes cube. Each extra axis is then given by keyword (`alpha=`) and interpolated like Teff; a text-valued axis would be matched exactly. Leaving it out raises `ValueError`, which names the axes.

In [ ]:
whole = SED.local("Kurucz2003all")
print(whole.model().extra_axes)
try:
    whole(5777, 4.44, 0.0)
except ValueError as error:
    print(error)

In [ ]:
alpha0 = whole(5777, 4.44, 0.0, alpha=0.0)
alpha4 = whole(5777, 4.44, 0.0, alpha=0.4)
alpha2 = whole(5777, 4.44, 0.0, alpha=0.2)

from_variant = variant(5777, 4.44, 0.0)
print("alpha=0 vs its variant folder:", np.max(np.abs(alpha0.flux - from_variant.flux)) / from_variant.flux.max())
assert np.allclose(alpha0.flux, from_variant.flux, rtol=1e-10, atol=0)

fig, ax = plt.subplots(figsize=(7, 3.5))
for s, label in ((alpha0, "alpha = 0"), (alpha2, "alpha = 0.2 (interpolated)"), (alpha4, "alpha = 0.4")):
    ax.plot(s.wavelength, s.flux / alpha0.flux, lw=0.8, label=label)
ax.set(xscale="log", xlim=(1500, 2e4), ylim=(0.5, 1.3), xlabel="Wavelength (Å)",
       ylabel="flux / flux(alpha = 0)", title="5777 K, log g 4.44, [M/H] 0")
ax.legend(fontsize=8);

Any extra axis works the same way. Husfeld's hot-star grid is split by helium abundance, `yhe`:

In [ ]:
husfeld = SED.local("Husfeld")
print(husfeld.parameter_ranges(), husfeld.model().extra_axes)
hot = husfeld(50000, 5.0, 0.0, yhe=0.1)
print(f"Husfeld 50000 K, yhe = 0.1: ∫F/σT⁴ = {np.trapezoid(hot.flux, hot.wavelength) / (SIGMA_SB * 50000**4):.3f}")

## 4. Writing a grid to another folder

`sed.grid.write(folder)` writes the grid's spectra with a lookup table, flux cube(s) and HDF5 bundle of their own. A split grid is written split again: its variant folders, `variants_index.csv` and `flux_cube_nd.bin`.
A selection made with `grid.filter(...)` can be written the same way, for example to use part of a grid in MESA.

Write to another folder: `write()` with no folder writes into the grid's own folder.

In [ ]:
out = SED._model_root / "Kurucz2003all_copy"
whole.grid.write(out)
print(sorted(p.name for p in out.iterdir() if not p.name.endswith(".txt")))
assert (out / "variants_index.csv").is_file() and (out / "flux_cube_nd.bin").is_file()
copy = SED.local("Kurucz2003all_copy")(5777, 4.44, 0.0, alpha=0.4)
assert np.allclose(copy.flux, alpha4.flux, rtol=1e-6)

## Command-line equivalents

| Python | Command line |
|---|---|
| `SED.local(name)` then `sed(...)` | `python -m sed_tools.cube.tool` (interpolate and photometer interactively) |
| `sed.grid.write(folder)` | `sed-tools rebuild --grids NAME` rebuilds a grid's lookup table, cubes and bundle in place |